# Phase 1 route capture — Colab runbook

This notebook does **not** close 108 routes. The previous draft incorrectly claimed CPU NumPy runs were proof for native/device/public-range routes and multimodal rows; those commands did not execute the claimed backends. Current capture support is compiled-native text routes for standalone Inkling, Qwen4, Qwen3.5, and GLM5; exact Inkling, Qwen4, and GLM5 text-device routes; and Qwen4/Inkling text public-range routes. GLM5 native compares against Torch; GLM5 device compares against the independently compiled C decoder.

These runners cover at most three native text rows per matching identity (F32/F16/BF16), not each family’s 21-row block. Inkling, Qwen4, and GLM5 standalone text also have device capture paths. They do not cover Qwen4 publisher multimodal rows; Inkling’s `inkling_mm_model` MTP/media rows; GLM5 publisher media/public-range rows; or other device/public-range rows. Upload the exact working tree and route-matching checkpoints. Receipts count only after `phase1_verify.py` accepts measured contents; Colab Pro does not guarantee a particular accelerator.

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile

drive.mount('/content/drive')
# Create the zip with the repository contents at its top level (phase1_verify.py
# should be at the root), and include all uncommitted source changes.
REPO_ZIP = Path('/content/drive/MyDrive/openagent-phase1.zip')
REPO_ROOT = Path('/content/openagent')
if not REPO_ZIP.is_file():
    raise FileNotFoundError(f'Upload the current D:/openagent working tree as {REPO_ZIP}')
if REPO_ROOT.exists():
    raise FileExistsError(f'{REPO_ROOT} already exists; use a fresh Colab runtime')
with zipfile.ZipFile(REPO_ZIP) as archive:
    root = REPO_ROOT.resolve()
    if any(not (root / name).resolve().is_relative_to(root) for name in archive.namelist()):
        raise ValueError('repository zip contains a path outside its extraction root')
    archive.extractall(root)
if not (REPO_ROOT / 'phase1_verify.py').is_file():
    raise ValueError('zip must contain the repository root, including phase1_verify.py')
print('Working tree:', REPO_ROOT)

In [ ]:
%cd /content/openagent
!pip -q install -e '.[engine]' safetensors
!python -c "import numpy, safetensors; print('capture dependencies ready')"

## Supply exact checkpoint directories

Each directory must have a config with the exact model identity expected by the route and all checkpoint shards at the root. The runner rechecks source tensor dtype/inventory and rejects mismatches before writing a receipt. Provide one checkpoint per source format; do not relabel a single checkpoint as F32/F16/BF16. Dense `qwen3_5` is not the admitted Qwen3.5-MoE identity.

In [ ]:
from pathlib import Path
import os, subprocess, sys

# Replace these with exact qwen4_exp_text checkpoints whose tensor inventory
# is homogeneous in the named source dtype. Leave unavailable formats out.
QWEN4_CHECKPOINTS = {
    # 'F32': Path('/content/drive/MyDrive/checkpoints/qwen4-f32'),
    # 'F16': Path('/content/drive/MyDrive/checkpoints/qwen4-f16'),
    # 'BF16': Path('/content/drive/MyDrive/checkpoints/qwen4-bf16'),
}
PROMPT_IDS = '1,3,5,7'
LIBRARY_DIR = Path('/content/openagent-native')
RECEIPTS_OUT = Path('/content/openagent/phase1_runtime_receipts.json')
EVIDENCE_OUT = Path('/content/drive/MyDrive/phase1_evidence')
EVIDENCE_OUT.mkdir(parents=True, exist_ok=True)
compiled = subprocess.run([sys.executable, '-c',
    'from openagent.adaptive_engine.native_family_capture import compile_native_text_kernel; '
    f'print(compile_native_text_kernel(\"qwen4\", {str(LIBRARY_DIR)!r}))'],
    cwd='/content/openagent', text=True, capture_output=True)
if compiled.returncode:
    raise RuntimeError(compiled.stdout + compiled.stderr)
QWEN4_LIBRARY = Path(compiled.stdout.strip().splitlines()[-1])
print('Compiled candidate:', QWEN4_LIBRARY)

results = []
for source_format, checkpoint in QWEN4_CHECKPOINTS.items():
    if not checkpoint.is_dir():
        raise FileNotFoundError(checkpoint)
    command = [sys.executable, '-m', 'openagent.adaptive_engine.phase1_capture',
        'qwen4', '--checkpoint', str(checkpoint), '--model-type', 'qwen4_exp_text',
        '--source-format', source_format, '--backend', 'native',
        '--library', str(QWEN4_LIBRARY), '--prompt-ids', PROMPT_IDS,
        '--evidence-root', str(EVIDENCE_OUT), '--receipts', str(RECEIPTS_OUT)]
    result = subprocess.run(command, cwd='/content/openagent', text=True, capture_output=True)
    print(source_format, 'PASS' if result.returncode == 0 else 'FAIL')
    if result.returncode:
        print(result.stdout, result.stderr)
    else:
        results.append(source_format)
print('Successful exact Qwen4 text captures:', results)

In [ ]:
# Exact qwen4_exp_text device route; uses Torch on the actual accelerator.
QWEN4_DEVICE_CHECKPOINTS = QWEN4_CHECKPOINTS
device_results = []
for source_format, checkpoint in QWEN4_DEVICE_CHECKPOINTS.items():
    command = [sys.executable, '-m', 'openagent.adaptive_engine.phase1_capture',
        'qwen4', '--checkpoint', str(checkpoint), '--model-type', 'qwen4_exp_text',
        '--source-format', source_format, '--backend', 'device', '--device', 'cuda:0',
        '--prompt-ids', PROMPT_IDS, '--evidence-root', str(EVIDENCE_OUT),
        '--receipts', str(RECEIPTS_OUT)]
    result = subprocess.run(command, cwd='/content/openagent', text=True, capture_output=True)
    print(source_format, 'PASS' if result.returncode == 0 else 'FAIL')
    if result.returncode:
        print(result.stdout, result.stderr)
    else:
        device_results.append(source_format)
print('Successful exact Qwen4 text device captures:', device_results)

In [ ]:
# Exact Qwen4 text through the Segment/Edge public dispatcher. Both compiled libraries are pinned in each receipt.
from openagent.adaptive_engine.device_capture import compile_segment_runtime
QWEN4_PUBLIC_BUILD = Path('/content/openagent-segment-runtime')
QWEN4_PUBLIC_LIBRARY = compile_segment_runtime(QWEN4_PUBLIC_BUILD)
public_results = []
for source_format, checkpoint in QWEN4_CHECKPOINTS.items():
    command = [sys.executable, '-m', 'openagent.adaptive_engine.phase1_capture',
        'qwen4', '--checkpoint', str(checkpoint), '--model-type', 'qwen4_exp_text',
        '--source-format', source_format, '--backend', 'public-range',
        '--library', str(QWEN4_LIBRARY), '--runtime-library', str(QWEN4_PUBLIC_LIBRARY),
        '--prompt-ids', PROMPT_IDS, '--evidence-root', str(EVIDENCE_OUT),
        '--receipts', str(RECEIPTS_OUT)]
    result = subprocess.run(command, cwd='/content/openagent', text=True, capture_output=True)
    print(source_format, 'PUBLIC PASS' if result.returncode == 0 else 'PUBLIC FAIL')
    if result.returncode:
        print(result.stdout, result.stderr)
    else:
        public_results.append(source_format)
print('Successful exact Qwen4 public-range text captures:', public_results)

In [ ]:
# Standalone inkling_text only; inkling_mm_model has MTP/media obligations.
INKLING_CHECKPOINTS = {
    # 'F32': Path('/content/drive/MyDrive/checkpoints/inkling-text-f32'),
    # 'F16': Path('/content/drive/MyDrive/checkpoints/inkling-text-f16'),
    # 'BF16': Path('/content/drive/MyDrive/checkpoints/inkling-text-bf16'),
}
compiled = subprocess.run([sys.executable, '-c',
    'from openagent.adaptive_engine.native_family_capture import compile_native_text_kernel; '
    f'print(compile_native_text_kernel(\"inkling\", {str(LIBRARY_DIR)!r}))'],
    cwd='/content/openagent', text=True, capture_output=True)
if compiled.returncode:
    raise RuntimeError(compiled.stdout + compiled.stderr)
INKLING_LIBRARY = Path(compiled.stdout.strip().splitlines()[-1])
results = []
for source_format, checkpoint in INKLING_CHECKPOINTS.items():
    command = [sys.executable, '-m', 'openagent.adaptive_engine.phase1_capture',
        'inkling', '--checkpoint', str(checkpoint), '--source-format', source_format,
        '--backend', 'native', '--library', str(INKLING_LIBRARY),
        '--prompt-ids', PROMPT_IDS, '--evidence-root', str(EVIDENCE_OUT),
        '--receipts', str(RECEIPTS_OUT)]
    result = subprocess.run(command, cwd='/content/openagent', text=True, capture_output=True)
    print(source_format, 'PASS' if result.returncode == 0 else 'FAIL')
    if result.returncode:
        print(result.stdout, result.stderr)
    else:
        results.append(source_format)
print('Successful exact Inkling text captures:', results)

In [ ]:
# Exact inkling_text device route; this identity has no media/MTP obligation.
INKLING_DEVICE_CHECKPOINTS = INKLING_CHECKPOINTS
device_results = []
for source_format, checkpoint in INKLING_DEVICE_CHECKPOINTS.items():
    command = [sys.executable, '-m', 'openagent.adaptive_engine.phase1_capture',
        'inkling', '--checkpoint', str(checkpoint), '--source-format', source_format,
        '--backend', 'device', '--device', 'cuda:0', '--prompt-ids', PROMPT_IDS,
        '--evidence-root', str(EVIDENCE_OUT), '--receipts', str(RECEIPTS_OUT)]
    result = subprocess.run(command, cwd='/content/openagent', text=True, capture_output=True)
    print(source_format, 'PASS' if result.returncode == 0 else 'FAIL')
    if result.returncode:
        print(result.stdout, result.stderr)
    else:
        device_results.append(source_format)
print('Successful exact Inkling text device captures:', device_results)

In [ ]:
# Exact inkling_text through public Segment/Edge; MTP/MM identity is intentionally excluded.
from openagent.adaptive_engine.device_capture import compile_segment_runtime
INKLING_PUBLIC_BUILD = Path('/content/openagent-inkling-segment-runtime')
INKLING_PUBLIC_LIBRARY = compile_segment_runtime(INKLING_PUBLIC_BUILD)
public_results = []
for source_format, checkpoint in INKLING_CHECKPOINTS.items():
    command = [sys.executable, '-m', 'openagent.adaptive_engine.phase1_capture',
        'inkling', '--checkpoint', str(checkpoint), '--source-format', source_format,
        '--backend', 'public-range', '--library', str(INKLING_LIBRARY),
        '--runtime-library', str(INKLING_PUBLIC_LIBRARY), '--prompt-ids', PROMPT_IDS,
        '--evidence-root', str(EVIDENCE_OUT), '--receipts', str(RECEIPTS_OUT)]
    result = subprocess.run(command, cwd='/content/openagent', text=True, capture_output=True)
    print(source_format, 'PUBLIC PASS' if result.returncode == 0 else 'PUBLIC FAIL')
    if result.returncode:
        print(result.stdout, result.stderr)
    else:
        public_results.append(source_format)
print('Successful exact Inkling public-range text captures:', public_results)

In [ ]:
# Native GLM5 text candidate, independently compared on the selected accelerator.
GLM5_TEXT_CHECKPOINTS = {
    # 'F32': Path('/content/drive/MyDrive/checkpoints/glm5-text-f32'),
    # 'F16': Path('/content/drive/MyDrive/checkpoints/glm5-text-f16'),
    # 'BF16': Path('/content/drive/MyDrive/checkpoints/glm5-text-bf16'),
}
compiled = subprocess.run([sys.executable, '-c',
    'from openagent.adaptive_engine.native_family_capture import compile_native_text_kernel; '
    f'print(compile_native_text_kernel(\"glm5-next\", {str(LIBRARY_DIR)!r}))'],
    cwd='/content/openagent', text=True, capture_output=True)
if compiled.returncode:
    raise RuntimeError(compiled.stdout + compiled.stderr)
GLM5_LIBRARY = Path(compiled.stdout.strip().splitlines()[-1])
results = []
for source_format, checkpoint in GLM5_TEXT_CHECKPOINTS.items():
    command = [sys.executable, '-m', 'openagent.adaptive_engine.phase1_capture',
        'glm5', '--checkpoint', str(checkpoint), '--source-format', source_format,
        '--backend', 'native', '--reference-device', 'cuda:0',
        '--library', str(GLM5_LIBRARY), '--prompt-ids', PROMPT_IDS,
        '--evidence-root', str(EVIDENCE_OUT), '--receipts', str(RECEIPTS_OUT)]
    result = subprocess.run(command, cwd='/content/openagent', text=True, capture_output=True)
    print(source_format, 'PASS' if result.returncode == 0 else 'FAIL')
    if result.returncode:
        print(result.stdout, result.stderr)
    else:
        results.append(source_format)
print('Successful exact GLM5 text native captures:', results)
device_results = []
for source_format, checkpoint in GLM5_TEXT_CHECKPOINTS.items():
    command = [sys.executable, '-m', 'openagent.adaptive_engine.phase1_capture',
        'glm5', '--checkpoint', str(checkpoint), '--source-format', source_format,
        '--backend', 'device', '--device', 'cuda:0',
        '--reference-library', str(GLM5_LIBRARY), '--prompt-ids', PROMPT_IDS,
        '--evidence-root', str(EVIDENCE_OUT), '--receipts', str(RECEIPTS_OUT)]
    result = subprocess.run(command, cwd='/content/openagent', text=True, capture_output=True)
    print(source_format, 'DEVICE PASS' if result.returncode == 0 else 'DEVICE FAIL')
    if result.returncode:
        print(result.stdout, result.stderr)
    else:
        device_results.append(source_format)
print('Successful exact GLM5 text device captures:', device_results)

In [ ]:
# Verification is authoritative: it reopens every pinned artifact/observation.
verification = subprocess.run([sys.executable, 'phase1_verify.py'],
    cwd='/content/openagent', text=True, capture_output=True)
print(verification.stdout)
if verification.returncode:
    print(verification.stderr)
    raise RuntimeError('Phase 1 remains open; do not count rejected receipts')

## Remaining work toward 108

The 108 figure is 45 OLMoE + 21 Inkling + 21 GLM5-Next + 21 Qwen4 route rows. Capture code now covers 24/63 non-OLMoE target rows: native and device text routes for standalone Inkling, Qwen4, and GLM5 (18), plus Qwen4 and Inkling public-range text for F32/F16/BF16 (6). No fresh receipts are claimed by this notebook. Remaining target rows still require Inkling speculative RNG plus `inkling_mm_model` media/public-range, GLM5 publisher media/public-range, and Qwen4 publisher multimodal capture (39 rows in total). Real matching checkpoints and actual backend execution/placement are required; do not count notebook setup or a successful build as evidence.